In [ ]:
import pandas as pd
import numpy as np
!pip install boto3
import boto3
from sklearn.model_selection import train_test_split
import io
import os

In [ ]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.6 MB/s eta 0:00:00


In [ ]:
conn = boto3.client('s3')

In [ ]:
# bucket='cs310-final-project-playlist-recommender'

In [ ]:
# content = conn.list_objects(Bucket=bucket)["Contents"]

In [ ]:
for data in content:
    print(data['Key'])

spotify-2023.csv


In [ ]:
dataset = 'spotify-2023.csv'

In [ ]:
df = pd.read_csv(dataset, encoding="latin1")

In [ ]:
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


In [ ]:
df.shape

(953, 24)

In [ ]:
df.describe()

,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,in_apple_playlists,in_apple_charts,in_deezer_charts,bpm,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
count,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.00000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000
mean,1.556139,2018.238195,6.033578,13.930745,5200.124869,12.009444,67.812172,51.908709,2.666317,122.540399,66.96957,51.431270,64.279119,27.057712,1.581322,18.213012,10.131165
std,0.893044,11.116218,3.566435,9.201949,7897.608990,19.575992,86.441493,50.630241,6.035599,28.057802,14.63061,23.480632,16.550526,25.996077,8.409800,13.711223,9.912888
min,1.000000,1930.000000,1.000000,1.000000,31.000000,0.000000,0.000000,0.000000,0.000000,65.000000,23.00000,4.000000,9.000000,0.000000,0.000000,3.000000,2.000000
25%,1.000000,2020.000000,3.000000,6.000000,875.000000,0.000000,13.000000,7.000000,0.000000,100.000000,57.00000,32.000000,53.000000,6.000000,0.000000,10.000000,4.000000
50%,1.000000,2022.000000,6.000000,13.000000,2224.000000,3.000000,34.000000,38.000000,0.000000,121.000000,69.00000,51.000000,66.000000,18.000000,0.000000,12.000000,6.000000
75%,2.000000,2022.000000,9.000000,22.000000,5542.000000,16.000000,88.000000,87.000000,2.000000,140.000000,78.00000,70.000000,77.000000,43.000000,0.000000,24.000000,11.000000
max,8.000000,2023.000000,12.000000,31.000000,52898.000000,147.000000,672.000000,275.000000,58.000000,206.000000,96.00000,97.000000,97.000000,97.000000,91.000000,97.000000,64.000000


In [ ]:
df.columns

Index(['track_name', 'artist(s)_name', 'artist_count', 'released_year',
       'released_month', 'released_day', 'in_spotify_playlists',
       'in_spotify_charts', 'streams', 'in_apple_playlists', 'in_apple_charts',
       'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts', 'bpm',
       'key', 'mode', 'danceability_%', 'valence_%', 'energy_%',
       'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
track_name,0
artist(s)_name,0
artist_count,0
released_year,0
released_month,0
released_day,0
in_spotify_playlists,0
in_spotify_charts,0
streams,0
in_apple_playlists,0


In [ ]:
df = df[df["artist_count"] == 1]

In [ ]:
df["mode"] = df["mode"].map({"Minor": 0, "Major": 1})

In [ ]:
df["track_name"] = df["track_name"].str.strip().str.lower()
df["artist(s)_name"] = df["artist(s)_name"].str.strip().str.lower()

In [ ]:
feature_cols = [
    "bpm",
    "mode",
    "danceability_%",
    "valence_%",
    "energy_%",
    "acousticness_%",
    "instrumentalness_%",
    "liveness_%",
    "speechiness_%"
]

In [ ]:
X = df[feature_cols].copy()

In [ ]:
# scales data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# trains model
from sklearn.neighbors import NearestNeighbors

nn_model = NearestNeighbors(
    n_neighbors=11,   # 1 will usually be the song itself, so ask for 11 to return 10 others
    metric="euclidean"
)

nn_model.fit(X_scaled)

NearestNeighbors(metric='euclidean', n_neighbors=11)

In [ ]:
track_name = "vampire"
artist_name = "Olivia Rodrigo"

In [ ]:
match = df[
    (df["track_name"].str.lower() == track_name.lower()) &
    (df["artist(s)_name"].str.lower().str.contains(artist_name.lower(), na=False))
]

In [ ]:
# locates song in dataset
def find_song(song_name, artist_name, df):
    song_name = song_name.strip().lower()
    artist_name = artist_name.strip().lower()

    match = df[
        (df["track_name"] == song_name) &
        (df["artist(s)_name"].str.contains(artist_name, na=False))
    ]

    return match

In [ ]:
def recommend_songs(song_name, artist_name, df, X_scaled, nn_model, n_recs=10):
    match = find_song(song_name, artist_name, df)

    if match.empty:
        return {
            "found_in_dataset": False,
            "recommendations": []
        }

    song_idx = match.index[0]

    distances, indices = nn_model.kneighbors(
        [X_scaled[song_idx]],
        n_neighbors=n_recs + 1
    )

    neighbor_indices = indices[0][1:]   # skip the song itself
    neighbor_distances = distances[0][1:]

    recs = df.iloc[neighbor_indices][["track_name", "artist(s)_name"]].copy()
    recs["distance"] = neighbor_distances

    return {
        "found_in_dataset": True,
        "input_song": {
            "track_name": df.loc[song_idx, "track_name"],
            "artist(s)_name": df.loc[song_idx, "artist(s)_name"]
        },
        "recommendations": recs.to_dict(orient="records")
    }

In [ ]:
song_name = "People"
artist_name = "Libianca"
result = recommend_songs(song_name, artist_name, df, X_scaled, nn_model)
print(result)

{'found_in_dataset': True, 'input_song': {'track_name': 'people', 'artist(s)_name': 'libianca'}, 'recommendations': [{'track_name': 'celestial', 'artist(s)_name': 'ed sheeran', 'distance': 0.7036832956370238}, {'track_name': 'sweater weather', 'artist(s)_name': 'the neighbourhood', 'distance': 0.7926886563023041}, {'track_name': 'you belong with me (taylorï¿½ï¿½ï¿½s ve', 'artist(s)_name': 'taylor swift', 'distance': 0.8079512944438942}, {'track_name': 'acapulco', 'artist(s)_name': 'jason derulo', 'distance': 0.8440997061760988}, {'track_name': 'take my breath', 'artist(s)_name': 'the weeknd', 'distance': 0.8837314281923412}, {'track_name': 'just the way you are', 'artist(s)_name': 'bruno mars', 'distance': 0.8903414611863163}, {'track_name': 'pepas', 'artist(s)_name': 'farruko', 'distance': 0.8988560633183872}, {'track_name': 'dandelions', 'artist(s)_name': 'ruth b.', 'distance': 0.9025753151017992}, {'track_name': 'adore you', 'artist(s)_name': 'harry styles', 'distance': 1.0359298922